# 10_make_hard_profile_dataset_colab

v6A 第一步：从已同步到 Google Drive 项目目录的 `lora_mvp` clean 音频派生 hard-profile train/val 数据。这个 notebook 只生成数据和统计，不跑 base inference，也不开始训练。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

default_project_dir = Path("/content/drive/MyDrive/qwen3-asr") if IN_COLAB else Path.cwd()
PROJECT_DIR = Path(os.environ.get("QWEN_ASR_PROJECT_DIR", str(default_project_dir))).expanduser()
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}. Run notebook 00_clone_github_colab first.")

os.chdir(PROJECT_DIR)
print("PROJECT_DIR =", PROJECT_DIR)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=False)
subprocess.run(["git", "log", "-1", "--oneline"], check=False)

In [ ]:
CONFIG = Path("configs/data/v6a_hard_profile.yaml")
SCRIPT = Path("scripts/create_v6a_hard_profile_dataset.py")
SOURCE_TRAIN = Path("data/jsonl/lora_mvp_train.local.jsonl")
SOURCE_VAL = Path("data/jsonl/lora_mvp_val.local.jsonl")
SOURCE_AUDIO = Path("data/lora_mvp/audio")

required = [CONFIG, SCRIPT, SOURCE_TRAIN, SOURCE_VAL, SOURCE_AUDIO]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required v6A inputs:\n" + "\n".join(missing))

print("v6A preflight passed")

In [ ]:
SMOKE_DIR = Path("/tmp/mega-asr-v6a-smoke")
smoke_cmd = [
    sys.executable,
    str(SCRIPT),
    "--config", str(CONFIG),
    "--max-train-base-utterances", "2",
    "--max-val-base-utterances", "1",
    "--variants-per-utterance", "1",
    "--output-dir", str(SMOKE_DIR / "audio"),
    "--train-manifest", str(SMOKE_DIR / "train.jsonl"),
    "--val-manifest", str(SMOKE_DIR / "val.jsonl"),
    "--stats", str(SMOKE_DIR / "stats.json"),
    "--force",
]
subprocess.run(smoke_cmd, check=True)

smoke_stats = json.loads((SMOKE_DIR / "stats.json").read_text())
assert smoke_stats["rows"]["train"] == 14, smoke_stats["rows"]
assert smoke_stats["rows"]["val"] == 7, smoke_stats["rows"]
assert smoke_stats["validation"]["missing_audio"] == 0
assert smoke_stats["validation"]["train_val_base_utterance_overlap"] == 0
assert smoke_stats["validation"]["train_val_source_base_utterance_overlap"] == 0
print("v6A smoke passed")
print(json.dumps(smoke_stats["scenario_counts"], ensure_ascii=False, indent=2))

In [ ]:
RUN_FULL_GENERATION = True
FORCE_REBUILD = True

if RUN_FULL_GENERATION:
    full_cmd = [sys.executable, str(SCRIPT), "--config", str(CONFIG)]
    if FORCE_REBUILD:
        full_cmd.append("--force")
    subprocess.run(full_cmd, check=True)
else:
    print("RUN_FULL_GENERATION=False, skipped full v6A generation")

In [ ]:
STATS = Path("data/jsonl/v6a_hard_profile_stats.local.json")
TRAIN_MANIFEST = Path("data/jsonl/v6a_hard_profile_train.local.jsonl")
VAL_MANIFEST = Path("data/jsonl/v6a_hard_profile_val.local.jsonl")

stats = json.loads(STATS.read_text())
assert stats["rows"]["train"] == stats["row_expectations"]["expected_train_rows"], stats["rows"]
assert stats["rows"]["val"] == stats["row_expectations"]["expected_val_rows"], stats["rows"]
assert stats["validation"]["missing_audio"] == 0
assert stats["validation"]["forbidden_path_hits"] == 0
assert stats["validation"]["train_val_base_utterance_overlap"] == 0
assert stats["validation"]["train_val_source_base_utterance_overlap"] == 0

print("v6A hard-profile data ready")
print("train manifest:", TRAIN_MANIFEST)
print("val manifest:", VAL_MANIFEST)
print("stats:", STATS)
print(json.dumps({
    "rows": stats["rows"],
    "source_clean_rows": stats["source_clean_rows"],
    "scenario_counts": stats["scenario_counts"],
    "next_steps": stats["next_steps"],
}, ensure_ascii=False, indent=2))